# Nước nào tăng GDP mà giảm CO2 - thật, hay chỉ "xuất khẩu" phát thải?

Bộ `owid-co2` (Our World in Data, CC BY 4.0, ghim commit `382ee6c`). Cột dùng trong bài:

| Cột | Nghĩa |
| --- | --- |
| `co2` | CO2 **sản xuất** (phát thải trên lãnh thổ), triệu tấn |
| `consumption_co2` | CO2 **tiêu dùng**: sản xuất + phát thải của hàng nhập khẩu − phát thải của hàng xuất khẩu (có từ 1990) |
| `gdp` | GDP thực, đô la quốc tế giá 2011 (có tới 2022) |
| `population` | dân số |

**Bối cảnh.** Một quỹ đầu tư hỏi team phân tích: *"Có bao nhiêu nền kinh tế lớn đã tách được tăng trưởng khỏi phát
thải (GDP tăng, CO2 giảm) từ 2005 tới 2022? Hay họ chỉ chuyển nhà máy sang nước khác?"*. "Tách rời" (decoupling) có
hai mức:

- **absolute**: GDP tăng, phát thải **giảm**;
- **relative**: GDP tăng, phát thải tăng **chậm hơn** GDP;
- ngược lại là **coupled** (phát thải tăng nhanh bằng hoặc hơn GDP); GDP không tăng thì là **gdp_decline**.

So phân loại theo CO2 sản xuất và theo CO2 tiêu dùng: nước "absolute" theo sản xuất mà không "absolute" theo tiêu dùng
là dấu hiệu phát thải được chuyển ra nước ngoài qua hàng nhập khẩu.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/04-python-for-analysts/exercises/intermediate/02-co2-decoupling -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc CSV
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

co2 = pd.read_csv(ensure("owid-co2") / "owid-co2-data.csv")
for col in ["co2", "consumption_co2", "gdp", "population"]:
    years = co2.loc[co2[col].notna() & co2.iso_code.notna(), "year"]
    print(f"{col:16s} có dữ liệu {years.min()}-{years.max()}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

COLS = ["co2", "consumption_co2", "gdp"]
CLASSES = ["absolute", "relative", "coupled", "gdp_decline"]

## 1. Thay đổi giữa hai năm, mỗi nước một dòng

`period_change(df, start, end, cols, min_population) -> pd.DataFrame`:

- Chỉ dùng dòng quốc gia (`iso_code` khác rỗng).
- Ghép ảnh chụp năm `start` với ảnh chụp năm `end` theo `country` (một nước một dòng mỗi năm: dùng `merge` có
  `validate="one_to_one"`).
- Giữ nước có **mọi** cột trong `cols` khác rỗng ở **cả hai** năm, và `population` năm `end` `>= min_population`.
- Trả về index `country` (sắp theo tên), mỗi cột trong `cols` (đúng thứ tự) là tỉ lệ thay đổi `end / start - 1`.

In [ ]:
def period_change(df: pd.DataFrame, start: int, end: int, cols: list[str],
                  min_population: float = 10e6) -> pd.DataFrame:
    """Index country; mỗi cột trong cols là end / start - 1; lọc dữ liệu đủ và dân số năm end."""
    nations = df[df["iso_code"].notna()]
    keep = ["country", *cols]
    a = nations.loc[nations["year"] == start, keep]
    b = nations.loc[nations["year"] == end, [*keep, "population"]]
    both = a.merge(b, on="country", suffixes=("_start", "_end"), validate="one_to_one")
    needed = [f"{c}_{s}" for c in cols for s in ("start", "end")]
    both = both.dropna(subset=needed)
    both = both[both["population"] >= min_population]
    out = pd.DataFrame({c: both[f"{c}_end"].to_numpy() / both[f"{c}_start"].to_numpy() - 1 for c in cols},
                       index=pd.Index(both["country"], name="country"))
    return out.sort_index()

In [ ]:
# explore: 2005 -> 2022, nước từ 10 triệu dân
change = period_change(co2, 2005, 2022, COLS)
print(len(change), "nước")
print(change.sort_values("co2").head(8).round(3).to_string())
print(change.loc[["Vietnam", "China", "India", "United States"]].round(3).to_string())

## 2. Phân loại decoupling

`classify(change, emissions="co2") -> pd.Series`: mỗi nước một nhãn theo thứ tự ưu tiên:

1. `gdp <= 0` → `"gdp_decline"`;
2. `emissions < 0` → `"absolute"`;
3. `emissions < gdp` → `"relative"`;
4. còn lại → `"coupled"`.

`emissions` là tên cột phát thải (`"co2"` hoặc `"consumption_co2"`). Series cùng index với `change`, kiểu chuỗi,
tên `decoupling`.

`decoupling_table(change) -> pd.DataFrame`: bản sao của `change` thêm ba cột `production` (= `classify` theo `co2`),
`consumption` (= `classify` theo `consumption_co2`), `offshoring` (`True` khi `production == "absolute"` nhưng
`consumption != "absolute"`).

In [ ]:
def classify(change: pd.DataFrame, emissions: str = "co2") -> pd.Series:
    """absolute / relative / coupled / gdp_decline theo cột gdp và cột emissions; tên decoupling."""
    gdp, em = change["gdp"], change[emissions]
    labels = np.select([gdp <= 0, em < 0, em < gdp], ["gdp_decline", "absolute", "relative"], default="coupled")
    return pd.Series(labels, index=change.index, name="decoupling").astype(str)


def decoupling_table(change: pd.DataFrame) -> pd.DataFrame:
    """change + production, consumption, offshoring."""
    out = change.copy()
    out["production"] = classify(change, "co2")
    out["consumption"] = classify(change, "consumption_co2")
    out["offshoring"] = (out["production"] == "absolute") & (out["consumption"] != "absolute")
    return out

In [ ]:
# explore: bảng chéo sản xuất x tiêu dùng, và các nước "absolute" chỉ theo sản xuất
table = decoupling_table(change)
print(table["production"].value_counts().reindex(CLASSES).to_string())
print(pd.crosstab(table["production"], table["consumption"]).reindex(index=CLASSES, columns=CLASSES, fill_value=0))
print(table[table.offshoring].round(3).to_string())
print(table[table.production == "absolute"].sort_values("co2").round(3).to_string())

In [ ]:
# explore: scatter GDP vs CO2 sản xuất, màu theo nhãn; đường chéo = tăng bằng nhau
plot = table.reset_index()
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=plot, x="gdp", y="co2", hue="production", hue_order=CLASSES, ax=ax)
lim = [-0.7, 3.5]
ax.plot(lim, lim, ls="--", color="grey", lw=1)
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlim(*lim)
ax.set_ylim(-0.7, 9)
for name in ["United Kingdom", "Vietnam", "China", "Poland", "United States"]:
    ax.annotate(name, (table.loc[name, "gdp"], table.loc[name, "co2"]), fontsize=8, xytext=(4, 4),
                textcoords="offset points")
ax.set_xlabel("GDP thực 2005 → 2022 (tỉ lệ thay đổi)")
ax.set_ylabel("CO2 sản xuất 2005 → 2022 (tỉ lệ thay đổi)")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("17 nước tăng GDP mà giảm CO2 sản xuất (góc phải dưới)", loc="left")
plt.show()

In [ ]:
# explore: sản xuất vs tiêu dùng cho nhóm absolute - mũi tên từ CO2 sản xuất tới CO2 tiêu dùng
abs_ = table[table.production == "absolute"].sort_values("co2")
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(abs_.co2 * 100, abs_.index, color="#4c78a8", label="sản xuất")
ax.scatter(abs_.consumption_co2 * 100, abs_.index, color="#e45756", label="tiêu dùng")
ax.hlines(abs_.index, abs_.co2 * 100, abs_.consumption_co2 * 100, color="#c0c7d0")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("% thay đổi CO2 2005 → 2022")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Ba nước giảm CO2 sản xuất nhưng CO2 tiêu dùng vẫn tăng", loc="left")
plt.show()

In [ ]:
# explore: kết quả nhạy với lựa chọn nào? Đổi năm đầu và ngưỡng dân số
for start, pop in [(2005, 10e6), (2008, 10e6), (2010, 10e6), (2005, 1e6)]:
    t = decoupling_table(period_change(co2, start, 2022, COLS, min_population=pop))
    print(f"{start}-2022, dân số >= {pop / 1e6:.0f} triệu: {len(t)} nước, absolute sản xuất "
          f"{(t.production == 'absolute').sum()}, absolute cả tiêu dùng "
          f"{((t.production == 'absolute') & (t.consumption == 'absolute')).sum()}")

## 3. Đọc kết quả, rồi làm trắc nghiệm

Viết cho quỹ đầu tư một đoạn 4–5 câu:

1. Trong bao nhiêu nước từ 10 triệu dân, bao nhiêu nước tách rời tuyệt đối theo CO2 sản xuất, bao nhiêu nước giữ được
   khi tính theo CO2 tiêu dùng? Những nước nào rơi ra?
2. Việt Nam, Trung Quốc, Ấn Độ nằm ở nhóm nào?
3. Kết quả nhạy với năm đầu và ngưỡng dân số thế nào (cell cuối)? Bạn sẽ ghi giới hạn gì?
4. Phân tích này **không** nói được điều gì (gợi ý: nhân quả, chính sách, GDP 2022 sau COVID)?

Xong thì mở app, làm phần **trắc nghiệm** của bài.